<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자기지도 학습 · 멀티모달 · 04. DINO: label 없는 self-distillation

## Emerging Properties in Self-Supervised Vision Transformers

- **원 논문:** [Emerging Properties in Self-Supervised Vision Transformers](https://arxiv.org/abs/2104.14294)
- **저자 / 발표:** Mathilde Caron et al. · ICCV (2021)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** knowledge distillation, EMA, temperature softmax
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** student/teacher, sharpening, centering, EMA를 결합한 DINO self-distillation을 구현합니다.

**축소하거나 생략한 부분:** ImageNet ViT multi-crop 대신 8×8 이미지의 두 global view와 MLP를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3, Figure 2 | 동일 image의 다른 view student/teacher | cross-view loss |
| Eq. (1)–(2) | temperature softmax와 cross entropy | 확률 합 |
| Eq. (3) | teacher parameter EMA | stop-gradient |
| §3.2 및 Figure 3 | centering+sharpening collapse 방지 | entropy·center 변화 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all,Path,np.load,torch.tensor,torch.float32 -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L=\tfrac12[H(P_t(x_2),P_s(x_1))+H(P_t(x_1),P_s(x_2))]$$

$$P_t(x)=\operatorname{softmax}((g_{\theta_t}(x)-c)/\tau_t)$$

- **기호 정의:** P_s/P_t는 student/teacher 분포, c는 center, τ_t는 teacher 온도입니다.
- **수식의 역할:** centering·sharpening EMA teacher를 반대 view의 student에 대칭 증류합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3, Figure 2을 구현합니다.
- **입출력 shape:** 두 image view [B, 1, 8, 8] → prototype 확률 [B, K]
- **평가:** cross-view CE, teacher entropy, center 및 EMA update
- **원문 대비 한계:** ImageNet ViT multi-crop 대신 8×8 이미지의 두 global view와 MLP를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Figure 2
- **이 셀의 책임:** 동일 image의 다른 view student/teacher
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** cross-view loss. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,nn.GELU -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def dino_net(out=24):
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 48), nn.GELU(), nn.Linear(48, out))


def dino_view(x):
    return (
        x * (0.75 + 0.5 * torch.rand(len(x), 1, 1, 1, device=x.device))
        + 0.03 * torch.randn_like(x)
    ).clamp(0, 1)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Eq. (1)–(2) / §3.2 및 Figure 3
- **이 셀의 책임:** temperature softmax와 cross entropy / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 확률 합 / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.log_softmax,torch.softmax,torch.randn,torch.zeros,torch.allclose,Tensor.sum,torch.ones -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.log_softmax(input, dim, dtype=None)</code></summary>

#### `F.log_softmax(input, dim, dtype=None)`

- **역할:** softmax의 로그를 수치적으로 안정된 한 연산으로 계산합니다.
- **입력·반환:** logit과 축을 받고 같은 shape의 log-probability를 반환합니다.
- **이 셀에서 쓰는 이유:** NLL, KL divergence, log-likelihood 계산에 사용합니다.
- **주의점:** `log(softmax(x))`를 따로 계산하면 overflow나 underflow가 생길 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [F.log_softmax](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.
- **관련 수식:** 이 셀의 `F.log_softmax(input, dim, dtype=None)`와 같은 수식 계약을 사용합니다. 중복 표시는 생략했습니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 04-1: centering과 서로 다른 temperature의 student/teacher 분포를 만드세요.
def dino_distributions(
    student_logits, teacher_logits, center, student_t=0.1, teacher_t=0.04
):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** Eq. (1)–(2) / Eq. (3)
- **이 셀의 책임:** temperature softmax와 cross entropy / teacher parameter EMA
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 확률 합 / stop-gradient. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.no_grad,Module.parameters,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 04-2: cross-view distillation CE와 teacher EMA를 구현하세요.
def distill_ce(student_log, teacher_prob):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


@torch.no_grad()
def update_teacher(student, teacher, momentum):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3, Figure 2 / Eq. (1)–(2) / Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** 동일 image의 다른 view student/teacher / temperature softmax와 cross entropy / teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** cross-view loss / 확률 합 / stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=ACCELERATOR.move,torch.optim.Adam,Optimizer.zero_grad,Tensor.backward,Optimizer.step,torch.cat,Tensor.detach,np.isfinite,torch.linalg.norm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 04-3: DINO update와 center EMA를 학습하고 teacher entropy를 추적하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3, Figure 2 / Eq. (1)–(2) / Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** 동일 image의 다른 view student/teacher / temperature softmax와 cross entropy / teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** cross-view loss / 확률 합 / stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,nn.ReLU,Tensor.requires_grad_,Module.register_buffer,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>tensor.requires_grad_(requires_grad=True)</code></summary>

#### `tensor.requires_grad_(requires_grad=True)`

- **역할:** 텐서의 gradient 추적 여부를 in-place로 바꿉니다.
- **입력·반환:** 불리언을 받고 같은 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** teacher 동결이나 새로 학습할 파라미터 범위를 명시합니다.
- **주의점:** 이미 만들어진 계산 그래프를 소급해 바꾸지 않으며 leaf 여부를 확인해야 합니다.
- **공식 문서:** [Tensor.requires_grad_](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.requires_grad_.html)

</details>

<details>
<summary><code>module.register_buffer(name, tensor, persistent=True)</code></summary>

#### `module.register_buffer(name, tensor, persistent=True)`

- **역할:** 학습하지 않지만 state와 장치 이동을 따르는 텐서를 등록합니다.
- **입력·반환:** 이름·텐서·저장 여부를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** position 상수나 running state를 파라미터와 구분해 보관합니다.
- **주의점:** 일반 attribute와 달리 state dict 포함 여부가 `persistent`에 좌우됩니다.
- **공식 문서:** [Module.register_buffer](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO P-1: 논문의 modality 준비, 핵심 module, objective와 평가를 구현하세요.
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    hidden_dim: int = 16
    student_temperature: float = 0.2
    teacher_temperature: float = 0.07
    momentum: float = 0.9
    center_momentum: float = 0.9
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class DinoPortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(
        self, first: Tensor, second: Tensor
    ) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(
        self,
        student_first: Tensor,
        student_second: Tensor,
        teacher_first: Tensor,
        teacher_second: Tensor,
    ) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    @torch.no_grad()
    def update_center(self, teacher_first: Tensor, teacher_second: Tensor) -> None:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def update_target(self) -> None:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(self, batch: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: DinoPortfolioModel, batch: Tensor) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** §3, Figure 2
- **이 셀의 책임:** 동일 image의 다른 view student/teacher
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** cross-view loss. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.normalize,Tensor.std,Tensor.cpu -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.normalize(input, p=2.0, dim=1, eps=1e-12)</code></summary>

#### `F.normalize(input, p=2.0, dim=1, eps=1e-12)`

- **역할:** 지정 축의 벡터 norm을 1로 정규화합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\hat x=\frac{x}{\max(\lVert x\rVert_p,\epsilon)}
$$

- **기호:** $p$는 norm 차수, $\epsilon$은 0으로 나누지 않기 위한 하한입니다.
- **수식 → 코드:** 코드의 `dim` 축에서 norm을 구한 뒤 모든 원소를 같은 값으로 나눕니다.
- **직관:** 벡터 크기를 제거해 방향이나 상대 패턴에 집중하게 합니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택 축의 norm은 약 1입니다.
- **공식 문서:** [F.normalize](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)

</details>

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def evaluate_portfolio_model(
    model: DinoPortfolioModel, batch: Tensor
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.